# 05 - Model Provenance Verification Demo

Companion notebook to `06-data-compliance-and-model-governance.md`. Implements the checksum/provenance
verification pattern chapter 06 describes: hash a synthetic "model weights" file, compare it against a
"published reference" hash, and demonstrate both a clean match and a tampered/mismatched case being
caught before deployment.

The "model weights" here are a small synthetic binary file standing in for a real multi-gigabyte
checkpoint -- the hashing and comparison logic is the real, general-purpose mechanism chapter 06
describes, just applied to a toy-sized artifact so the notebook runs instantly and offline. No real
model weights, no network calls.

## 1. Simulate publishing a model artifact and its reference hash

Mistral (the publisher) releases a model artifact and, alongside it, a reference SHA-256 hash anyone can
use to verify their downloaded copy is bit-for-bit identical to what was actually published -- chapter
06's "checksumming against the publisher's published model card" step.

In [1]:
import hashlib
import random

def sha256_of_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

# Simulate the publisher's artifact: a small synthetic "weights" blob (real Mixtral weights are
# many gigabytes; this stands in for them so the notebook runs instantly, offline).
random.seed(13)
PUBLISHED_WEIGHTS = bytes(random.getrandbits(8) for _ in range(4096))
PUBLISHED_REFERENCE_HASH = sha256_of_bytes(PUBLISHED_WEIGHTS)

print("Publisher (Mistral AI) releases a model artifact.")
print("Published artifact size: {} bytes (synthetic stand-in for a real multi-GB checkpoint)".format(len(PUBLISHED_WEIGHTS)))
print("Published reference SHA-256: {}".format(PUBLISHED_REFERENCE_HASH))


Publisher (Mistral AI) releases a model artifact.
Published artifact size: 4096 bytes (synthetic stand-in for a real multi-GB checkpoint)
Published reference SHA-256: e248ce0845aa10d71b6c3a6a9902728cd4a4222a6f9174ab8b1e849614e2c092


## 2. Case A: a clean, genuine download -- verification passes

The platform downloads the artifact from an approved distribution channel. Before it's ever loaded into
the serving layer (chapter 04) or used as a fine-tuning base (chapter 05), it's hashed and compared
against the publisher's reference hash.

In [2]:
def verify_artifact(downloaded_bytes, reference_hash):
    computed_hash = sha256_of_bytes(downloaded_bytes)
    match = (computed_hash == reference_hash)
    return {
        "computed_sha256": computed_hash,
        "reference_sha256": reference_hash,
        "verification_status": "MATCH" if match else "MISMATCH",
    }


# Case A: a genuine, uncorrupted download -- bit-for-bit identical to what was published.
downloaded_genuine = PUBLISHED_WEIGHTS  # a real download would be a separate byte-for-byte copy; identical content here

result_a = verify_artifact(downloaded_genuine, PUBLISHED_REFERENCE_HASH)
print("Case A -- genuine download:")
for k, v in result_a.items():
    print("  {}: {}".format(k, v))

assert result_a["verification_status"] == "MATCH"
print()
print("MATCH -- safe to deploy. This is the registry record chapter 06 describes being carried")
print("durably alongside the artifact.")


Case A -- genuine download:
  computed_sha256: e248ce0845aa10d71b6c3a6a9902728cd4a4222a6f9174ab8b1e849614e2c092
  reference_sha256: e248ce0845aa10d71b6c3a6a9902728cd4a4222a6f9174ab8b1e849614e2c092
  verification_status: MATCH

MATCH -- safe to deploy. This is the registry record chapter 06 describes being carried
durably alongside the artifact.


## 3. Case B: a tampered/corrupted checkpoint -- verification catches it

Now simulate the scenario chapter 06 is actually worried about: a checkpoint that's been modified after
publication -- whether through corruption, a compromised mirror, or a deliberately tampered
redistribution -- flowing into the same verification step. Even a single flipped byte is enough to
completely change the hash, which is exactly why hash verification is such an effective, cheap check.

In [3]:
# Case B: tamper with a SINGLE byte of the artifact -- simulating either corruption in transit or a
# deliberately modified checkpoint redistributed under the same name.
tampered_weights = bytearray(PUBLISHED_WEIGHTS)
tamper_index = 2048
tampered_weights[tamper_index] = (tampered_weights[tamper_index] + 1) % 256  # flip one byte
tampered_weights = bytes(tampered_weights)

result_b = verify_artifact(tampered_weights, PUBLISHED_REFERENCE_HASH)
print("Case B -- tampered download (single byte changed at offset {}):".format(tamper_index))
for k, v in result_b.items():
    print("  {}: {}".format(k, v))

assert result_b["verification_status"] == "MISMATCH"
bytes_changed = sum(1 for a, b in zip(PUBLISHED_WEIGHTS, tampered_weights) if a != b)
print()
print("MISMATCH -- caught. Only {} byte(s) out of {} were changed, but the hash is completely different".format(
    bytes_changed, len(PUBLISHED_WEIGHTS)))
print("from the reference. Per chapter 06: a mismatch is a hard stop, not a warning -- this artifact")
print("must NOT be deployed until the discrepancy is understood.")


Case B -- tampered download (single byte changed at offset 2048):
  computed_sha256: 8e1d6966fea7a623dd1c9c73d22a84e2c6ac114a4fd04978019b6df2ef7434c2
  reference_sha256: e248ce0845aa10d71b6c3a6a9902728cd4a4222a6f9174ab8b1e849614e2c092
  verification_status: MISMATCH

MISMATCH -- caught. Only 1 byte(s) out of 4096 were changed, but the hash is completely different
from the reference. Per chapter 06: a mismatch is a hard stop, not a warning -- this artifact
must NOT be deployed until the discrepancy is understood.


## 4. The durable provenance record (chapter 06's registry design)

Whatever the verification outcome, it's recorded durably in the model registry entry for the artifact --
not just checked once and discarded. This is the record chapter 07 later extends with a
`taxonomy_version` tag.

In [4]:
import json
from datetime import datetime, timezone

def build_registry_record(artifact_id, verify_result, source, downloaded_by):
    return {
        "artifact_id": artifact_id,
        "base_model_weights_sha256": verify_result["computed_sha256"],
        "publisher_reference_hash": verify_result["reference_sha256"],
        "verification_status": verify_result["verification_status"],
        "hash_verified_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source": source,
        "downloaded_by": downloaded_by,
    }


record_a = build_registry_record(
    artifact_id="mixtral-8x7b-base-v1",
    verify_result=result_a,
    source="Mistral AI official release channel",
    downloaded_by="platform-mlops-service-principal",
)
record_b = build_registry_record(
    artifact_id="mixtral-8x7b-base-v1-SUSPECT",
    verify_result=result_b,
    source="third-party mirror (unverified)",
    downloaded_by="platform-mlops-service-principal",
)

print("Registry record -- Case A (clean download):")
print(json.dumps(record_a, indent=2))
print()
print("Registry record -- Case B (tampered download):")
print(json.dumps(record_b, indent=2))

assert record_a["verification_status"] == "MATCH"
assert record_b["verification_status"] == "MISMATCH"
print()
print("Both outcomes are recorded durably -- chapter 06's point that verification isn't a one-time gate,")
print("it's a standing, auditable record any model-risk reviewer can inspect later without re-deriving it.")


Registry record -- Case A (clean download):
{
  "artifact_id": "mixtral-8x7b-base-v1",
  "base_model_weights_sha256": "e248ce0845aa10d71b6c3a6a9902728cd4a4222a6f9174ab8b1e849614e2c092",
  "publisher_reference_hash": "e248ce0845aa10d71b6c3a6a9902728cd4a4222a6f9174ab8b1e849614e2c092",
  "verification_status": "MATCH",
  "hash_verified_at": "2026-07-28T18:27:14+00:00",
  "source": "Mistral AI official release channel",
  "downloaded_by": "platform-mlops-service-principal"
}

Registry record -- Case B (tampered download):
{
  "artifact_id": "mixtral-8x7b-base-v1-SUSPECT",
  "base_model_weights_sha256": "8e1d6966fea7a623dd1c9c73d22a84e2c6ac114a4fd04978019b6df2ef7434c2",
  "publisher_reference_hash": "e248ce0845aa10d71b6c3a6a9902728cd4a4222a6f9174ab8b1e849614e2c092",
  "verification_status": "MISMATCH",
  "hash_verified_at": "2026-07-28T18:27:14+00:00",
  "source": "third-party mirror (unverified)",
  "downloaded_by": "platform-mlops-service-principal"
}

Both outcomes are recorded durably -

## Summary

| Section | Demonstrates | Matches |
|---|---|---|
| 1 | A publisher's reference hash alongside a released artifact | Chapter 06's "hash published alongside the model card" |
| 2 | A genuine download verified as a clean match | Chapter 06's normal, expected path |
| 3 | A single-byte tampering caught by hash mismatch | Chapter 06's core supply-chain-integrity concern |
| 4 | A durable, structured registry record for both outcomes | Chapter 06's provenance-tracking design, extended by chapter 07's taxonomy-version tag |

The sharpest point this notebook makes concrete: changing a **single byte** out of thousands completely
changes the SHA-256 hash, which is exactly why hash verification is such a cheap, effective check --
and exactly why skipping it before deploying a downloaded open-weight checkpoint into a regulated
institution's production infrastructure is a real, avoidable risk, not a theoretical one (chapter 06).